### Librerías a utilizar
---

In [1]:
import pickle
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import  root_mean_squared_error
from sklearn.feature_extraction import  DictVectorizer
from sklearn.preprocessing import StandardScaler
import mlflow
from dotenv import load_dotenv
import math
import optuna
import pathlib
from optuna.samplers import TPESampler
from mlflow.models.signature import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from mlflow import MlflowClient
from datetime import datetime
import mlflow.pyfunc as mlflow_pyfunc
from statsmodels.stats.outliers_influence import variance_inflation_factor
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split


c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Cargar las credenciales
---

In [2]:
load_dotenv(override=True)  # Carga las variables del archivo .env
EXPERIMENT_NAME = "/Users/monica.ibarra@iteso.mx/project1-experiment" 

mlflow.set_tracking_uri("databricks")
experiment = mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

### Preprocessing
___

In [ ]:
#def preprocessing(df: pd.DataFrame):
    # Quitar columnas
    #df = df.drop(columns=['Health_Issues', 'Caffeine_mg'], errors='ignore')

    # Filtrar filas con género "Other"
    #df = df[df["Gender"] != "Other"]

    # Mapear países a continentes 
    #pais_a_continente = {
        #"Canada": "America", "USA": "America", "Mexico": "America", "Brazil": "America",
        #"Norway": "Europe", "Sweden": "Europe", "UK": "Europe", "Finland": "Europe",
        #"Italy": "Europe", "Belgium": "Europe", "Germany": "Europe", "France": "Europe",
        #"Switzerland": "Europe", "Netherlands": "Europe", "Spain": "Europe",
        #"India": "Asia", "China": "Asia", "South Korea": "Asia", "Japan": "Asia",
        #"Australia": "Oceania"
    #}
    #df["Continent"] = df["Country"].map(pais_a_continente)

    # Mapear variables categóricas 
    #df['Sleep_Quality'] = df['Sleep_Quality'].map({'Poor': 0, 'Fair': 1, 'Good': 2, 'Excellent': 3})
    #df['Stress_Level'] = df['Stress_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})
    #df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})

    # Eliminar columnas
    #df = df.drop(columns=["Country"], errors='ignore')
    #if "ID" in df.columns:
        #df = df.drop(columns=["ID"])

    # Convertir booleanos a enteros 
    #for col in df.columns:
        #if df[col].dtype == 'bool':
            #df[col] = df[col].astype(int)

    # Convertir a matriz con DictVectorizer 
    #dicts = df.drop(columns=["Stress_Level"]).to_dict(orient="records")
    #dv = DictVectorizer(sparse=False)
    #X = dv.fit_transform(dicts)

    # Target 
    #y = df["Stress_Level"].values

    # Balancear con SMOTE 
    #smote = SMOTE(random_state=42)
    #X_bal, y_bal = smote.fit_resample(X, y)

    #print("Preprocessing completado.")
    #print("Shape features balanceadas:", X_bal.shape)
    #print("Distribución target balanceado:\n", pd.Series(y_bal).value_counts())

    #return X_bal, y_bal, dv

In [3]:
def preprocessing_train(df: pd.DataFrame, vif_threshold=5):
    # Mantener la misma firma mínima que pediste (solo lo necesario)
    df = df.drop(columns=['Health_Issues', 'Caffeine_mg', 'Sleep_Hours', 'Sleep_Quality'], errors='ignore')
    df = df[df["Gender"] != "Other"]

    pais_a_continente = {
        "Canada": "America", "USA": "America", "Mexico": "America", "Brazil": "America",
        "Norway": "Europe", "Sweden": "Europe", "UK": "Europe", "Finland": "Europe",
        "Italy": "Europe", "Belgium": "Europe", "Germany": "Europe", "France": "Europe",
        "Switzerland": "Europe", "Netherlands": "Europe", "Spain": "Europe",
        "India": "Asia", "China": "Asia", "South Korea": "Asia", "Japan": "Asia",
        "Australia": "Oceania"
    }
    df["Continent"] = df["Country"].map(pais_a_continente)

    df['Stress_Level'] = df['Stress_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})
    df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})

    df = df.drop(columns=["Country"], errors='ignore')
    if "ID" in df.columns:
        df = df.drop(columns=["ID"])

    for col in df.columns:
        if df[col].dtype == 'bool':
            df[col] = df[col].astype(int)

    # Separar X e y
    y = df["Stress_Level"].values
    X_df = df.drop(columns=["Stress_Level"])

    # DictVectorizer
    dicts = X_df.to_dict(orient="records")
    dv = DictVectorizer(sparse=False)
    X = dv.fit_transform(dicts)
    X_df_encoded = pd.DataFrame(X, columns=dv.get_feature_names_out())

    # --- Limpieza mínima antes de VIF ---
    # 1) reemplazar inf por NaN y luego rellenar NaN con 0
    X_df_encoded = X_df_encoded.replace([np.inf, -np.inf], np.nan)
    if X_df_encoded.isna().any().any():
        X_df_encoded = X_df_encoded.fillna(0)

    # 2) eliminar una dummy por cada categoría para evitar colinealidad perfecta
    feature_names = dv.get_feature_names_out().tolist()
    groups = {}
    for f in feature_names:
        if '=' in f:
            pref = f.split('=')[0]
            groups.setdefault(pref, []).append(f)
    to_drop = []
    for pref, feats in groups.items():
        if len(feats) > 1:
            feats_sorted = sorted(feats)
            to_drop.append(feats_sorted[0])
    if to_drop:
        X_df_encoded = X_df_encoded.drop(columns=[c for c in to_drop if c in X_df_encoded.columns], errors='ignore')

    # 3) eliminar columnas constantes
    nunique = X_df_encoded.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()
    if constant_cols:
        X_df_encoded = X_df_encoded.drop(columns=constant_cols, errors='ignore')

    # --- Filtrado VIF (defensivo) ---
    features = list(X_df_encoded.columns)
    dropped = []
    while True:
        if len(features) < 2:
            break
        vif_vals = []
        for i in range(len(features)):
            try:
                v = variance_inflation_factor(X_df_encoded[features].values, i)
                if not np.isfinite(v):
                    v = np.inf
            except Exception:
                v = np.inf
            vif_vals.append(v)

        vif_data = pd.DataFrame({"variable": features, "VIF": vif_vals})

        # eliminar VIF infinito primero
        if np.isinf(vif_data["VIF"]).any():
            variable_to_drop = vif_data.loc[vif_data["VIF"].idxmax(), "variable"]
            dropped.append((variable_to_drop, float('inf')))
            features.remove(variable_to_drop)
            print(f"VIF infinito -> Eliminando: {variable_to_drop}")
            continue

        max_vif = vif_data["VIF"].max()
        if max_vif > vif_threshold:
            variable_to_drop = vif_data.loc[vif_data["VIF"].idxmax(), "variable"]
            dropped.append((variable_to_drop, float(max_vif)))
            features.remove(variable_to_drop)
            print(f"Eliminando por VIF alto: {variable_to_drop} ({max_vif:.2f})")
        else:
            break

    # DataFrame con features finales (antes de scale)
    X_filtered_df = X_df_encoded[features]

    # --- Escalado (fit en train) ---
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_filtered_df.values)

    # --- SMOTE (usar matriz escalada) ---
    smote = SMOTE(random_state=42)
    X_bal, y_bal = smote.fit_resample(X_scaled, y)

    print("Preprocessing train completado.")
    print("Shape features balanceadas (train):", X_bal.shape)
    print("Distribución target balanceado (train):\n", pd.Series(y_bal).value_counts())

    # devolver scaler para usar en evaluación
    return X_bal, y_bal, dv, features, dropped, scaler

In [4]:
def preprocessing_eval(df: pd.DataFrame, dv: DictVectorizer, features, scaler: StandardScaler):
    # Versión mínima compatible + aplicación de scaler
    df = df.drop(columns=['Health_Issues', 'Caffeine_mg', 'Sleep_Hours', 'Sleep_Quality'], errors='ignore')
    df = df[df["Gender"] != "Other"]

    pais_a_continente = {
        "Canada": "America", "USA": "America", "Mexico": "America", "Brazil": "America",
        "Norway": "Europe", "Sweden": "Europe", "UK": "Europe", "Finland": "Europe",
        "Italy": "Europe", "Belgium": "Europe", "Germany": "Europe", "France": "Europe",
        "Switzerland": "Europe", "Netherlands": "Europe", "Spain": "Europe",
        "India": "Asia", "China": "Asia", "South Korea": "Asia", "Japan": "Asia",
        "Australia": "Oceania"
    }
    df["Continent"] = df["Country"].map(pais_a_continente)

    df['Stress_Level'] = df['Stress_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})
    df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})
    df = df.drop(columns=["Country"], errors='ignore')
    if "ID" in df.columns:
        df = df.drop(columns=["ID"])

    for col in df.columns:
        if df[col].dtype == 'bool':
            df[col] = df[col].astype(int)

    dicts = df.drop(columns=["Stress_Level"]).to_dict(orient="records")
    X_encoded = dv.transform(dicts).astype(float)

    X_encoded[~np.isfinite(X_encoded)] = np.nan
    if np.isnan(X_encoded).any():
        X_encoded = np.nan_to_num(X_encoded, nan=0.0, posinf=0.0, neginf=0.0)

    feature_names = dv.get_feature_names_out()
    X_df_encoded = pd.DataFrame(X_encoded, columns=feature_names)

    # eliminar mismas dummies base (primera dummy por prefijo)
    groups = {}
    for f in feature_names:
        if '=' in f:
            pref = f.split('=')[0]
            groups.setdefault(pref, []).append(f)
    to_drop = []
    for pref, feats in groups.items():
        if len(feats) > 1:
            feats_sorted = sorted(feats)
            to_drop.append(feats_sorted[0])
    if to_drop:
        X_df_encoded = X_df_encoded.drop(columns=[c for c in to_drop if c in X_df_encoded.columns], errors='ignore')

    # Asegurar que todas las features estén presentes (si falta alguna, añadir con 0)
    for f in features:
        if f not in X_df_encoded.columns:
            X_df_encoded[f] = 0.0

    # Seleccionar columnas en mismo orden que 'features'
    X_filtered_df = X_df_encoded[features]

    # Transformar con scaler (fue ajustado en train)
    X_scaled = scaler.transform(X_filtered_df.values)

    y = df["Stress_Level"].values
    return X_scaled, y

In [ ]:
#target = 'Stress_Level'  
#X = df.drop(columns=[target])
#y = df[target].values

### Dividir en entrenamiento, prueba & validacion
---

In [5]:
df_raw = pd.read_csv("../data/raw/synthetic_coffee_health_10000.csv")

In [6]:
target = 'Stress_Level'

In [7]:
train_df, temp_df = train_test_split(df_raw, test_size=0.4, random_state=42, stratify=df_raw[target])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df[target])

In [9]:
# Preprocess train (fit dv + SMOTE)
X_train_bal, y_train_bal, dv, features, dropped, scaler = preprocessing_train(train_df, vif_threshold=7)

Eliminando por VIF alto: Heart_Rate (29.62)
Eliminando por VIF alto: BMI (16.34)
Preprocessing train completado.
Shape features balanceadas (train): (12285, 13)
Distribución target balanceado (train):
 0    4095
2    4095
1    4095
Name: count, dtype: int64


In [10]:
print("Features seleccionadas (después de VIF):")
print(features)

Features seleccionadas (después de VIF):
['Age', 'Alcohol_Consumption', 'Coffee_Intake', 'Continent=Asia', 'Continent=Europe', 'Continent=Oceania', 'Gender', 'Occupation=Office', 'Occupation=Other', 'Occupation=Service', 'Occupation=Student', 'Physical_Activity_Hours', 'Smoking']


In [11]:
X_val, y_val = preprocessing_eval(val_df, dv, features, scaler)
X_test, y_test = preprocessing_eval(test_df, dv, features, scaler)

In [ ]:
#X_train, X_test_val, y_train, y_test_val = train_test_split(X, y, test_size=0.4, random_state=42)

#X_val, X_test, y_val, y_test = train_test_split(X_test_val, y_test_val, test_size=0.5, random_state=42)

### Regresión Logística
---

#### Función objetivo

In [12]:
def objective_logreg(trial: optuna.trial.Trial):
    # Hiperparámetros a buscar
    penalty = trial.suggest_categorical("penalty", ["l2", "l1", "elasticnet"])
    l1_ratio = None
    if penalty == "elasticnet":
        l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)

    # Construir param dict para logging y modelo
    params = {
        "penalty": penalty,
        "C": trial.suggest_float("C", math.exp(-7), 1e2, log=True),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "solver": "saga",           # saga soporta l1, elasticnet y multinomial
        "multi_class": "multinomial",
        "max_iter": 500,
        "random_state": 42,
        "n_jobs": -1
    }
    if penalty == "elasticnet":
        params["l1_ratio"] = l1_ratio
        params["penalty"] = "elasticnet"

    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "logistic_regression")
        mlflow.log_params(params)

        # Instanciar y entrenar
        model = LogisticRegression(**params)
        model.fit(X_train_bal, y_train_bal)

        # Validación
        y_proba = model.predict_proba(X_val)
        y_pred = model.predict(X_val)

        # Métricas (multiclase)
        val_logloss = log_loss(y_val, y_proba)
        val_acc = accuracy_score(y_val, y_pred)
        val_f1_macro = f1_score(y_val, y_pred, average="macro")

        # Log metrics
        mlflow.log_metric("accuracy", val_acc)
        mlflow.log_metric("f1_macro", val_f1_macro)

        # Guardar modelo (anidado)
        signature = infer_signature(X_val, y_proba)
        mlflow.sklearn.log_model(model, "model", input_example=X_val[:5], signature=signature)

    # Optuna minimiza -> devolvemos log_loss
    return val_logloss

#### Flujo de búsqueda

In [18]:
mlflow.sklearn.autolog(log_models=False)

sampler = TPESampler(seed=42)
study = optuna.create_study(direction="minimize", sampler=sampler)

with mlflow.start_run(run_name="LogisticRegression Hyperparameter Optimization (Optuna)", nested=True):
    study.optimize(objective_logreg, n_trials=10) 
    best_params = study.best_params
    mlflow.log_params(best_params)
    mlflow.set_tags({
        "project": "Stress Level Prediction",
        "optimizer_engine": "optuna",
        "model_family": "logistic_regression",
        "feature_set_version": 1
    })

final_params = {
    "solver": "saga",
    "multi_class": "multinomial",
    "max_iter": 500,
    "random_state": 42,
    "n_jobs": -1
}

# combinar best_params
final_params.update(best_params)

final_model_lr = LogisticRegression(**final_params)
final_model_lr.fit(X_train_bal, y_train_bal)

y_proba = final_model_lr.predict_proba(X_val)
y_pred = final_model_lr.predict(X_val)

val_logloss = log_loss(y_val, y_proba)
val_acc = accuracy_score(y_val, y_pred)
val_f1_macro = f1_score(y_val, y_pred, average="macro")

with mlflow.start_run(run_name="LogisticRegression - final", nested=False):
    mlflow.set_tag("model_family", "logistic_regression")
    mlflow.log_params(final_params)
    mlflow.log_metric("val_log_loss", val_logloss)
    mlflow.log_metric("val_accuracy", val_acc)
    mlflow.log_metric("val_f1_macro", val_f1_macro)

    # Guardar el preprocessor (dv)
    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

    # Log modelo final
    feature_names_final = features
    input_example = pd.DataFrame(X_val[:5], columns=feature_names_final)
    signature = infer_signature(input_example, y_val[:5])


    mlflow.sklearn.log_model(final_model_lr, "model", input_example=input_example, signature=signature)

print("Proceso completado.")
print("val log_loss:", val_logloss)
print("val accuracy:", val_acc)
print("val f1_macro:", val_f1_macro)

[I 2025-11-09 16:32:47,033] A new study created in memory with name: no-name-86493833-a317-432c-8815-971f1d4cdb57
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
2025/11/09 16:32:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 16:33:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/09 16:33:07 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environ

🏃 View run zealous-snipe-286 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/fd5a8670d31e4ed5b82399c995ab6213
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
2025/11/09 16:33:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 16:33:28 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/09 16:33:29 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-11-09 16:33:32,909] Trial 1 finished w

🏃 View run spiffy-hound-145 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/96d58e20522d43f0a4205a2e6553e38a
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
2025/11/09 16:33:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 16:33:48 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/09 16:33:50 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-11-09 16:33:54,053] Trial 2 finished w

...

🏃 View run able-bass-619 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/527742d769704f21bb3cf3ea9dfd9e0a
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794
🏃 View run LogisticRegression Hyperparameter Optimization (Optuna) at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/7322411dd3d14ead980a3e944c98667c
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/09 16:36:17 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '4cef6220c09f47f9b93db724759817de', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


🏃 View run delicate-lark-636 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/4cef6220c09f47f9b93db724759817de
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/09 16:36:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 16:36:35 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
2025/11/09 16:36:36 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


🏃 View run LogisticRegression - final at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/f8e6d68ed5ba4477b03b3c8619f6657f
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794
Proceso completado.
val log_loss: 1.077969449085923
val accuracy: 0.45184426229508196
val f1_macro: 0.34511499096561954


### Registrar modelo Champion
---

In [ ]:
# model_name = "workspace.default.equipo1-proyecto"

In [ ]:
#runs = mlflow.search_runs(
    #experiment_names=[EXPERIMENT_NAME],
    #order_by=["metrics.val_log_loss ASC"],
    #output_format="list"
#)

# Obtener el mejor run
#if len(runs) > 0:
#    best_run = runs[0]
#    print("🏆 Champion Run encontrado:")
#    print(f"Run ID: {best_run.info.run_id}")
#    print(f"Validation RMSE: {best_run.data.metrics.get('rmse')}")
#    print(f"Params: {best_run.data.params}")
#else:
#    print("⚠️ No se encontraron runs con métrica log_loss.")

In [ ]:
#run_id = best_run.info.run_id

In [ ]:
#result = mlflow.register_model(
    #model_uri=f"runs:/{best_run.info.run_id}/model",
    #name=model_name
#)

In [ ]:
#client = MlflowClient()

#model_version = result.version
#new_alias = "Champion"

#client.set_registered_model_alias(
    #name=model_name,
    #alias=new_alias,
    #version=result.version
#)

#date = datetime.today()

#client.update_model_version(
    #name=model_name,
    #version=model_version,
    #description=f"The model version {model_version} was transitioned to {new_alias} on {date}"
#)

### Random Forest
---

#### Función objetivo

In [14]:
# Requisitos: importados previamente
# import optuna
# from optuna.samplers import TPESampler
# import mlflow
# import pathlib, pickle
# from mlflow.models.signature import infer_signature
# from sklearn.metrics import log_loss, accuracy_score, f1_score
# from sklearn.ensemble import RandomForestClassifier
# import xgboost as xgb
# mlflow.sklearn.autolog(log_models=False)  # si no lo has hecho aún

# -------------------------
# Objective: Random Forest
# -------------------------
def objective_rf(trial: optuna.trial.Trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "random_state": 42,
        "n_jobs": -1
    }

    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "random_forest")
        mlflow.log_params(params)

        clf = RandomForestClassifier(**params)
        clf.fit(X_train_bal, y_train_bal)

        y_proba = clf.predict_proba(X_val)
        y_pred = clf.predict(X_val)

        val_logloss = log_loss(y_val, y_proba)
        val_acc = accuracy_score(y_val, y_pred)
        val_f1 = f1_score(y_val, y_pred, average="macro")

        mlflow.log_metric("val_log_loss", val_logloss)
        mlflow.log_metric("val_accuracy", val_acc)
        mlflow.log_metric("val_f1_macro", val_f1)

        # Guardar modelo del trial
        signature = infer_signature(X_val, y_val)

        mlflow.sklearn.log_model(clf, "model", input_example=X_val[:5], signature=signature)

    return val_logloss




#### Flujo de búsqueda

In [20]:
sampler = TPESampler(seed=42)
study_rf = optuna.create_study(direction="minimize", sampler=sampler)

with mlflow.start_run(run_name="RandomForest Hyperparameter Optimization (Optuna)", nested=True):
    study_rf.optimize(objective_rf, n_trials=10)
    best_rf = study_rf.best_params
    mlflow.log_params(best_rf)
    mlflow.set_tags({
        "project": "Stress Level Prediction",
        "optimizer_engine": "optuna",
        "model_family": "random_forest"
    })

# Entrenar RF final con best params y loguear en un run final
final_rf_params = best_rf.copy()
final_rf_params.update({"random_state": 42, "n_jobs": -1})
rf_final = RandomForestClassifier(**final_rf_params)
rf_final.fit(X_train_bal, y_train_bal)
y_val_proba = rf_final.predict_proba(X_val)
y_val_pred = rf_final.predict(X_val)
rf_val_logloss = log_loss(y_val, y_val_proba)
rf_val_acc = accuracy_score(y_val, y_val_pred)
rf_val_f1 = f1_score(y_val, y_val_pred, average="macro")

with mlflow.start_run(run_name="RandomForest - final", nested=False):
    mlflow.log_params(final_rf_params)
    mlflow.log_metric("val_log_loss", rf_val_logloss)
    mlflow.log_metric("val_accuracy", rf_val_acc)
    mlflow.log_metric("val_f1_macro", rf_val_f1)
    # log preprocessor artifact (if not already logged)
    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")
    # log model
    feature_names_final = features
    input_example = pd.DataFrame(X_val[:5], columns=feature_names_final)
    signature = infer_signature(input_example, y_val[:5])
    mlflow.sklearn.log_model(rf_final, "model", input_example=input_example, signature=signature)

print("Random Forest terminado. val_logloss:", rf_val_logloss, "acc:", rf_val_acc, "f1:", rf_val_f1)


[I 2025-11-09 16:38:16,955] A new study created in memory with name: no-name-175d6798-93a8-424a-a490-f7002f402c7a
2025/11/09 16:38:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 16:38:37 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/09 16:38:38 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-11-09 16:39:14,611] Trial 0 finished with value: 1.007921572859944 and parameters: {'n_estimators': 406, 'max_depth': 29, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt', 'class_weight': None}. Best is trial 0 with value: 1.007921572859944.


🏃 View run receptive-shad-635 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/cd75412882424f73a990ed7f686311c8
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/09 16:39:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 16:39:36 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/09 16:39:37 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-11-09 16:39:42,531] Trial 1 finished with value: 1.0714786404058505 and parameters: {'n_estimators': 723, 'max_depth': 3, 'min_samples_split': 20, 'min_samples_leaf': 17, 'max_features': 'sqrt', 'class_weight': 'balanced'}. Best is trial 0 with value: 1.007921572859944.


🏃 View run secretive-horse-950 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/28407bdf3d064e09a448196f2c0f4f32
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/09 16:39:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 16:40:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/09 16:40:08 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-11-09 16:40:40,960] Trial 2 finished with value: 1.009092425099837 and parameters: {'n_estimators': 460, 'max_depth': 11, 'min_samples_split': 13, 'min_samples_leaf': 3, 'max_features': None, 'class_weight': None}. Best is trial 0 with value: 1.007921572859944.


...

🏃 View run RandomForest Hyperparameter Optimization (Optuna) at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/c24f8867121d4f9e9ff36d61e88aa2ef
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/09 16:48:45 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '36e7bdd9b611459bbd3c52818a814040', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


🏃 View run sassy-snail-292 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/36e7bdd9b611459bbd3c52818a814040
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/09 16:49:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/09 16:49:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
2025/11/09 16:49:56 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


🏃 View run RandomForest - final at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/7e6150fbbbfe4dbba3ae655e9ca3cdf2
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794
Random Forest terminado. val_logloss: 0.9564616703862309 acc: 0.5532786885245902 f1: 0.35282643607975556


### XGBoost
---

#### Función objetivo

In [16]:

# -------------------------
# Objective: XGBoost
# -------------------------
def objective_xgb(trial: optuna.trial.Trial):
    # xgboost params (sklearn API)

    params = {
        "max_depth": trial.suggest_int("max_depth", 4, 100),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "learning_rate": trial.suggest_float("learning_rate", math.exp(-3), 1.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha",   math.exp(-5), math.exp(-1), log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", math.exp(-6), math.exp(-1), log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", math.exp(-1), math.exp(3), log=True),
        "objective": "reg:squarederror",  
        "seed": 42,                      
    }
    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "xgboost")
        # log params but avoid logging num_class/use_label_encoder maybe
        mlflow.log_params(params)

        clf = xgb.XGBClassifier(**params)
        clf.fit(X_train_bal, y_train_bal, eval_set=[(X_val, y_val)], verbose=False)

        y_proba = clf.predict_proba(X_val)
        y_pred = clf.predict(X_val)

        val_logloss = log_loss(y_val, y_proba)
        val_acc = accuracy_score(y_val, y_pred)
        val_f1 = f1_score(y_val, y_pred, average="macro")

        mlflow.log_metric("val_log_loss", val_logloss)
        mlflow.log_metric("val_accuracy", val_acc)
        mlflow.log_metric("val_f1_macro", val_f1)


        signature = infer_signature(X_test, y_test[:5])

        mlflow.xgboost.log_model(clf, artifact_path="model", input_example=X_test[:5], signature=signature)

    return val_logloss

#### Flujo de búsqueda

In [17]:
# XGBoost study
study_xgb = optuna.create_study(direction="minimize", sampler=sampler)

with mlflow.start_run(run_name="XGBoost Hyperparameter Optimization (Optuna)", nested=True):
    study_xgb.optimize(objective_xgb, n_trials=30)
    best_xgb = study_xgb.best_params
    mlflow.log_params(best_xgb)
    mlflow.set_tags({
        "project": "Stress Level Prediction",
        "optimizer_engine": "optuna",
        "model_family": "xgboost"
    })

# Entrenar XGB final con best params y loguear en un run final
final_xgb_params = best_xgb.copy()
final_xgb_params.update({
    "objective": "multi:softprob",
    "use_label_encoder": False,
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": 0,
    "num_class": len(np.unique(y_train_bal))
})
xgb_final = xgb.XGBClassifier(**final_xgb_params)
xgb_final.fit(X_train_bal, y_train_bal)
y_test_proba = xgb_final.predict_proba(X_test)
y_test_pred = xgb_final.predict(X_test)
xgb_test_logloss = log_loss(y_test, y_test_proba)
xgb_test_acc = accuracy_score(y_test, y_test_pred)
xgb_test_f1 = f1_score(y_test, y_test_pred, average="macro")

with mlflow.start_run(run_name="XGBoost - final", nested=False):
    mlflow.log_params(final_xgb_params)
    mlflow.log_metric("test_log_loss", xgb_test_logloss)
    mlflow.log_metric("test_accuracy", xgb_test_acc)
    mlflow.log_metric("test_f1_macro", xgb_test_f1)
    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")
    feature_names_final = features
    input_example = pd.DataFrame(X_test[:5], columns=feature_names_final)
    signature = infer_signature(input_example, y_test[:5])

    # Prefer mlflow.xgboost.log_model, fallback to sklearn flavor
    mlflow.xgboost.log_model(xgb_final, artifact_path="model", input_example=input_example, signature=signature)
    
print("XGBoost terminado. test_logloss:", xgb_test_logloss, "acc:", xgb_test_acc, "f1:", xgb_test_f1)


[I 2025-11-09 16:12:26,836] A new study created in memory with name: no-name-5d997f10-c546-4998-a1e3-9ffc3802109f
2025/11/09 16:12:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1115: UserWarning: [16:12:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)
2025/11/09 16:12:43 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1124: UserWarning: [16:12:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_ap

🏃 View run debonair-moth-816 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/d5969f4300b0467f89849b47c3569d97
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/09 16:12:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1115: UserWarning: [16:12:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)
2025/11/09 16:13:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1124: UserWarning: [16:13:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1511: Unknown file format: `xgb`. Using UBJSON (`ubj`) as a guess.
  self.get_booster().load_model(fname)
[I 

🏃 View run nebulous-carp-497 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/88e6009796b9486781b295daacaba3d1
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/09 16:13:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1115: UserWarning: [16:13:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)
2025/11/09 16:13:36 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1124: UserWarning: [16:13:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1511: Unknown file format: `xgb`. Using UBJSON (`ubj`) as a guess.
  self.get_booster().load_model(fname)


...

🏃 View run shivering-pug-921 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/fcce2b4786d3421f8c7a56044bbc7a3a
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


[I 2025-11-09 16:23:48,174] Trial 29 finished with value: 0.9268230302334527 and parameters: {'max_depth': 71, 'n_estimators': 325, 'learning_rate': 0.13651834670168875, 'gamma': 2.6190705152032647, 'reg_alpha': 0.1751722881931259, 'reg_lambda': 0.023394965532206366, 'min_child_weight': 2.140364026996279}. Best is trial 22 with value: 0.9007974620742264.


🏃 View run XGBoost Hyperparameter Optimization (Optuna) at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/487ff34278ac41fa85796e8e4d398454
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/09 16:23:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1115: UserWarning: [16:23:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)
2025/11/09 16:24:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1124: UserWarning: [16:24:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1511: Unknown file format: `xgb`. Using UBJSON (`ubj`) as a guess.
  self.get_booster().load_model(fname)


🏃 View run XGBoost - final at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/be97bd56805448bb8a696936d4cab16a
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794
XGBoost terminado. test_logloss: 0.9003752519350361 acc: 0.610230179028133 f1: 0.3324203924379911


### Registrar modelo Challenger
---

In [ ]:
#runs = mlflow.search_runs(
    #experiment_names=[EXPERIMENT_NAME],
    #order_by=["metrics.val_log_loss ASC"],
    #output_format="list"
#)

# Obtener el mejor run
#if len(runs) > 0:
#    best_run = runs[0]
#    print("🏆 Champion Run encontrado:")
#    print(f"Run ID: {best_run.info.run_id}")
#    print(f"Validation RMSE: {best_run.data.metrics.get('rmse')}")
#    print(f"Params: {best_run.data.params}")
#else:
#    print("⚠️ No se encontraron runs con métrica log_loss.")

In [ ]:
#run_id = best_run.info.run_id

In [ ]:
#result = mlflow.register_model(
    #model_uri=f"runs:/{best_run.info.run_id}/model",
    #name=model_name
#)

In [ ]:
#client = MlflowClient()

#model_version = result.version
#new_alias = "Challenger"

#client.set_registered_model_alias(
    #name=model_name,
    #alias=new_alias,
    #version=result.version
#)

#date = datetime.today()

#client.update_model_version(
    #name=model_name,
    #version=model_version,
    #description=f"The model version {model_version} was transitioned to {new_alias} on {date}"
#)